## Demonstração do Controller de Categoria

Este notebook demonstra o funcionamento do controller de categorias da API, cobrindo as seguintes operações:
- Criar categoria (`POST /categories/create`)
- Buscar todos os categorias ativos (`GET /categories/all`)
- Buscar categoria por ID (`GET /categories/{id}`)
- Atualizar categoria (`PATCH /categories/{id}/update`)
- Desativar categoria (`DELETE /categories/{id}/delete`)
- Restaurar categoria desativado (`POST /categories/{id}/restore`)
- Deletar permanentemente do banco (`DELETE /categories/{id}/force-delete`)


## Setup do Teste com FastAPI e TestClient

Import e configuração do FastAPI com o router `categories`.

In [2]:
from fastapi.testclient import TestClient
from fastapi import FastAPI

from categories.controller import categories_router
from user.controller import users_router
from category_types.controller import category_types_router

app = FastAPI()
app.include_router(categories_router)
app.include_router(users_router)
app.include_router(category_types_router)

client = TestClient(app)

## Criar Usuário e Tipo de Categoria de Teste

**Endpoints:** 
`POST /users/create` 
`POST /category_types/create` 


In [3]:
user_payload = {
    "first_name": "Alan",
    "last_name": "Gamer",
    "cpf": "961.456.040-18",
    "email": "agamer@example.com",
    "password": "Strong@Password1986",
    "manual_balance": 500.0
}

response = client.post("/users/create", json=user_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_user = response.json()
user_id = created_user["id"]

user_payload = {
    "name": "Gastos",
    "is_positive": False,
}

response = client.post("/category_types/create", json=user_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_category_type = response.json()
category_type_id = created_category_type["id"]

2025-06-13 18:53:49,114 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2025-06-13 18:53:49,118 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-06-13 18:53:49,120 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2025-06-13 18:53:49,121 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-06-13 18:53:49,125 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2025-06-13 18:53:49,127 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-06-13 18:53:49,131 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-13 18:53:49,135 INFO sqlalchemy.engine.Engine INSERT INTO users (id, first_name, last_name, email, cpf, password, manual_balance, created_at, updated_at, deleted_at) VALUES (%(id)s, %(first_name)s, %(last_name)s, %(email)s, %(cpf)s, %(password)s, %(manual_balance)s, %(created_at)s, %(updated_at)s, %(deleted_at)s)
2025-06-13 18:53:49,136 INFO sqlalchemy.engine.Engine [generated in 0.00127s] {'id': '0684c9ded2277092800099a3c11e6d63', 'first_name': 'Alan', 'last_name': 'Gamer', 'email': '

## Criar Categoria de Teste

**Endpoint:** `POST /categories/create`  
**Descrição:** Cria um novo categoria com os dados fornecidos no payload.

In [4]:
user_payload = {
    "user_id": user_id,
    "category_type_id": category_type_id,
    "name": "Gastos de Alan",
    "description": "Categoria de gastos de Alan Gamer"
}

response = client.post("/categories/create", json=user_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_category = response.json()
category_id = created_category["id"]

2025-06-13 18:53:55,021 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-13 18:53:55,028 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.first_name AS users_first_name, users.last_name AS users_last_name, users.email AS users_email, users.cpf AS users_cpf, users.password AS users_password, users.manual_balance AS users_manual_balance, users.created_at AS users_created_at, users.updated_at AS users_updated_at, users.deleted_at AS users_deleted_at 
FROM users 
WHERE users.id = %(id_1)s AND users.deleted_at IS NULL 
 LIMIT %(param_1)s
2025-06-13 18:53:55,029 INFO sqlalchemy.engine.Engine [generated in 0.00105s] {'id_1': '0684c9ded2277092800099a3c11e6d63', 'param_1': 1}
2025-06-13 18:53:55,033 INFO sqlalchemy.engine.Engine SELECT category_types.id AS category_types_id, category_types.name AS category_types_name, category_types.is_positive AS category_types_is_positive 
FROM category_types 
WHERE category_types.id = %(id_1)s 
 LIMIT %(param_1)s
2025-06-13 18:53:55,035

## Buscar Todos os Categorias Ativos

**Endpoint:** `GET /categories/all`  
**Descrição:** Retorna todos os categorias que estão ativos no sistema.

In [5]:
response = client.get("/categories/all")
print("Status:", response.status_code)
for user in response.json():
    print(user)

2025-06-13 18:54:03,075 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-13 18:54:03,080 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS categories_description, categories.created_at AS categories_created_at, categories.updated_at AS categories_updated_at, categories.deleted_at AS categories_deleted_at 
FROM categories
2025-06-13 18:54:03,081 INFO sqlalchemy.engine.Engine [generated in 0.00106s] {}
2025-06-13 18:54:03,088 INFO sqlalchemy.engine.Engine ROLLBACK
Status: 200
{'id': '068166b5-b2e1-7056-8000-f25d2174f5f9', 'user_id': '027339ad-5777-2ca2-8675-0a3a2ad47b31', 'category_type_id': '068165d7-1db8-74f4-8000-51632b7e378e', 'name': 'Investimentos do Luiz', 'description': 'Categoria de investimentos do Luiz', 'created_at': '2025-05-03T16:13:32', 'updated_at': '2025-05-03T16:13:32', 'deleted_at': 

## Buscar Categoria por ID

**Endpoint:** `GET /categories/{id}`  
**Descrição:** Retorna os dados de um categoria específico, identificado pelo ID.

In [16]:
response = client.get(f"/categories/{category_id}")
print("Status:", response.status_code)
print("Categoria:", response.json())

2025-06-13 18:55:40,435 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-13 18:55:40,440 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS categories_description, categories.created_at AS categories_created_at, categories.updated_at AS categories_updated_at, categories.deleted_at AS categories_deleted_at 
FROM categories 
WHERE categories.id = %(id_1)s AND categories.deleted_at IS NULL 
 LIMIT %(param_1)s
2025-06-13 18:55:40,441 INFO sqlalchemy.engine.Engine [cached since 92.58s ago] {'id_1': '0684c9df30a277378000911be43351cd', 'param_1': 1}
2025-06-13 18:55:40,445 INFO sqlalchemy.engine.Engine ROLLBACK
Status: 404
Categoria: {'detail': {'Category': 'Not found'}}


## Atualizar Dados do Categoria

**Endpoint:** `PATCH /categories/{id}/update`  
**Descrição:** Atualiza os campos fornecidos do categoria (ex: saldo manual).

In [12]:
update_payload = {
    "name": "Gastos de Alan com Festas"
}

response = client.patch(f"/categories/{category_id}/update", json=update_payload)
print("Status:", response.status_code)
print("Atualizado:", response.json())

2025-06-13 18:54:56,406 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-13 18:54:56,411 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS categories_description, categories.created_at AS categories_created_at, categories.updated_at AS categories_updated_at, categories.deleted_at AS categories_deleted_at 
FROM categories 
WHERE categories.id = %(id_1)s 
 LIMIT %(param_1)s
2025-06-13 18:54:56,412 INFO sqlalchemy.engine.Engine [cached since 41.62s ago] {'id_1': '0684c9df30a277378000911be43351cd', 'param_1': 1}
2025-06-13 18:54:56,416 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS categories_description, categories.creat

## Desativar Categoria (Soft Delete)

**Endpoint:** `DELETE /categories/{id}/delete`  
**Descrição:** Marca o categoria como desativado, sem removê-lo do banco de dados.

In [15]:
response = client.delete(f"/categories/{category_id}/delete")
print("Status:", response.status_code)
print("Categoria deletada:", response.json())

2025-06-13 18:55:29,882 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-13 18:55:29,888 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS categories_description, categories.created_at AS categories_created_at, categories.updated_at AS categories_updated_at, categories.deleted_at AS categories_deleted_at 
FROM categories 
WHERE categories.id = %(id_1)s 
 LIMIT %(param_1)s
2025-06-13 18:55:29,889 INFO sqlalchemy.engine.Engine [cached since 75.1s ago] {'id_1': '0684c9df30a277378000911be43351cd', 'param_1': 1}
2025-06-13 18:55:29,893 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS categories_description, categories.create

## Restaurar Categoria Desativada

**Endpoint:** `POST /categories/{id}/restore`  
**Descrição:** Restaura um categoria que foi previamente desativado.

In [11]:
response = client.post(f"/categories/{category_id}/restore")
print("Status:", response.status_code)
print("Restaurado:", response.json())

2025-06-13 18:54:46,609 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-13 18:54:46,625 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS categories_description, categories.created_at AS categories_created_at, categories.updated_at AS categories_updated_at, categories.deleted_at AS categories_deleted_at 
FROM categories 
WHERE categories.id = %(id_1)s AND categories.deleted_at IS NOT NULL 
 LIMIT %(param_1)s
2025-06-13 18:54:46,626 INFO sqlalchemy.engine.Engine [cached since 5.519s ago] {'id_1': '0684c9df30a277378000911be43351cd', 'param_1': 1}
2025-06-13 18:54:46,630 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS ca

## Deletar Categoria Permanentemente

**Endpoint:** `DELETE /categories/{id}/force-delete`  
**Descrição:** Deleta permanentemente o categoria do banco de dados (hard delete). O categoria precisa estar desativado antes.

In [17]:
response = client.delete(f"/categories/{category_id}/force-delete")
print("Status:", response.status_code)
print("Forçando a Deleção no banco de dados:", response.json())

2025-06-13 18:55:47,839 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-13 18:55:47,844 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS categories_description, categories.created_at AS categories_created_at, categories.updated_at AS categories_updated_at, categories.deleted_at AS categories_deleted_at 
FROM categories 
WHERE categories.id = %(id_1)s AND categories.deleted_at IS NOT NULL 
 LIMIT %(param_1)s
2025-06-13 18:55:47,845 INFO sqlalchemy.engine.Engine [cached since 66.73s ago] {'id_1': '0684c9ded2277092800099a3c11e6d63', 'param_1': 1}
2025-06-13 18:55:47,848 INFO sqlalchemy.engine.Engine ROLLBACK
Status: 404
Forçando a Deleção no banco de dados: {'detail': {'Category': 'Not found, Object may be deleted or does not exist'}}
